# Hierarchical Turn-Transformer NLU (Perception & Telemetry Layer for LLM Agents)
### Dual-Level Architecture: DistilBERT (Level 1) + Turn-Transformer (Level 2) + Trajectory Summarizer
This model serves as the **high-speed perception layer** in a Compound AI architecture, extracting rich structured telemetry for downstream LLM Agents:
1. **Level 1 (Utterance Perception)**: Instantaneous Emotion & Sentiment Nuances (14 flags), Subword NER Slots (19 BIO tags).
2. **Level 2 (Dialogue Context & Trajectory Tracking)**: 2-Layer Turn-Transformer with **Speaker Embeddings** (`0=Customer`, `1=Agent`).
   - **Context-Aware Intent & Category**: Discovers customer goal from the complete conversation trajectory ($d_t$).
   - **Trajectory Summary Head**: Classifies conversation momentum (`STABLE_INQUIRY`, `ESCALATING_FRICTION`, `CRITICAL_CHURN_RISK`, `DE_ESCALATING_RESOLVED`).
   - **Dynamic Escalation & Effort Regressors**: Real-time trajectory friction index in $[0.00, 1.00]$.
3. **LLM Agent Payload Output**: Emits structured JSON for downstream tool calling and reasoning.

In [ ]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import transformers
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} (CUDA {torch.version.cuda})')

## 1. Load Dataset & Construct Multi-Turn Trajectory Taxonomy

In [ ]:
CSV_PATH = r'../archive/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv'
df = pd.read_csv(CSV_PATH)

# 1. Level 1 Categories & Intents
CATEGORIES = sorted(df['category'].unique().tolist())
INTENTS = sorted(df['intent'].unique().tolist())
cat2id = {c: i for i, c in enumerate(CATEGORIES)}
intent2id = {intent: i for i, intent in enumerate(INTENTS)}
id2cat = {i: c for c, i in cat2id.items()}
id2intent = {i: intent for intent, i in intent2id.items()}

# 2. Linguistic Flags
FLAGS = ['B', 'L', 'Q', 'I', 'Z', 'M', 'C', 'K', 'E', 'P', 'W', 'N', 'S', 'V']
flag2id = {f: i for i, f in enumerate(FLAGS)}
pos_weights = []
for f in FLAGS:
    df[f'flag_{f}'] = df['flags'].apply(lambda x: 1.0 if f in str(x) else 0.0)
    pos = df[f'flag_{f}'].sum()
    neg = len(df) - pos
    w = min(15.0, max(1.0, neg / max(pos, 1.0)))
    pos_weights.append(w)
FLAG_POS_WEIGHTS = torch.tensor(pos_weights, dtype=torch.float)

# 3. Level 2 Trajectory Summary Classes (4 states)
TRAJECTORIES = [
    'STABLE_INQUIRY',          # Routine troubleshooting / standard back-and-forth
    'ESCALATING_FRICTION',      # Customer getting frustrated / repeating details
    'CRITICAL_CHURN_RISK',     # Profanity / severe hostility / churn threat
    'DE_ESCALATING_RESOLVED'   # Issue being fixed / customer satisfied
]
traj2id = {t: i for i, t in enumerate(TRAJECTORIES)}
id2traj = {i: t for t, i in traj2id.items()}

# 4. NER Slot Entities
raw_entities = set()
for inst in df['instruction']:
    for match in re.findall(r'\{\{([^}]+)\}\}', str(inst)):
        raw_entities.add(match)
ENTITIES = sorted(list(raw_entities))
ner_labels = ['O']
for ent in ENTITIES:
    ent_tag = ent.replace(' ', '_')
    ner_labels.extend([f'B-{ent_tag}', f'I-{ent_tag}'])
ner2id = {l: i for i, l in enumerate(ner_labels)}
id2ner = {i: l for l, i in ner2id.items()}

print(f'Categories ({len(CATEGORIES)}) | Intents ({len(INTENTS)})')
print(f'NER Classes ({len(ner2id)}) | Trajectory States ({len(TRAJECTORIES)}): {TRAJECTORIES}')

## 2. Context-Aware Multi-Turn Thread Generator & Dataset

In [ ]:
SYNTHETIC_ENTITIES = {
    'Order Number': lambda: f'ORD-{random.randint(10000, 99999)}',
    'Refund Amount': lambda: f'${random.randint(10, 500)}.{random.randint(10, 99)}',
    'Invoice Number': lambda: f'INV-{random.randint(10000, 99999)}',
    'Delivery City': lambda: random.choice(['New York', 'Chicago', 'London', 'Berlin', 'Austin']),
    'Delivery Country': lambda: random.choice(['USA', 'Canada', 'Germany', 'UK']),
    'Person Name': lambda: random.choice(['John Doe', 'Sarah Connor', 'Alex Smith']),
    'Account Category': lambda: random.choice(['Personal', 'Business', 'VIP']),
    'Account Type': lambda: random.choice(['Standard', 'Premium', 'Pro']),
    'Currency Symbol': lambda: '$'
}

def inject_synthetic_entities(text: str):
    pattern = re.compile(r'\{\{([^}]+)\}\}')
    spans = []
    new_text = ''
    last_end = 0
    for match in pattern.finditer(text):
        ent_name = match.group(1).strip()
        generator = SYNTHETIC_ENTITIES.get(ent_name, lambda: ent_name)
        val = generator()
        new_text += text[last_end:match.start()]
        start_char = len(new_text)
        new_text += val
        end_char = len(new_text)
        spans.append((start_char, end_char, ent_name.replace(' ', '_')))
        last_end = match.end()
    new_text += text[last_end:]
    return new_text, spans

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

AGENT_CANNED_REPLIES = [
    'Could you please confirm your registered account email and order ID?',
    'Please provide your details so I can check your account records.',
    'I understand. Could you clarify the issue in more detail so I can assist?',
    'Let me pull up your account records to check this for you.'
]

def build_multi_turn_dialogue(row):
    text_t1, spans_t1 = inject_synthetic_entities(str(row['instruction']))
    flags = str(row['flags'])
    intent = str(row['intent'])
    intent_clean = intent.replace('_', ' ')
    
    agent_reply = random.choice(AGENT_CANNED_REPLIES)
    
    if 'W' in flags: # Severe Anger / Swearing
        text_t3 = random.choice([
            f'I already provided that! Why is this taking so damn long, get me a supervisor for my {intent_clean}!',
            f'This is ridiculous, stop sending canned messages and fix my {intent_clean} right now!',
            f'Unacceptable service! Stop asking stupid questions and cancel it damn it!'
        ])
        traj_label = traj2id['CRITICAL_CHURN_RISK']
        esc_score = 0.85
        ces_score = 0.80
    elif 'M' in flags or 'E' in flags: # Friction / Distress
        text_t3 = random.choice([
            f'I have been waiting for hours, can you please assist with my {intent_clean}?',
            f'I really need this resolved today, please check on my {intent_clean} again.',
            f'Why is this taking so long? I already provided all details for my {intent_clean}.'
        ])
        traj_label = traj2id['ESCALATING_FRICTION']
        esc_score = 0.50
        ces_score = 0.55
    else: # Stable / Routine
        text_t3 = random.choice([
            f'Sure, my email is user@example.com regarding my {intent_clean}.',
            f'Thanks, I have provided the details for my {intent_clean} as requested.',
            'Okay, let me know what the diagnostic check shows.'
        ])
        traj_label = traj2id['STABLE_INQUIRY']
        esc_score = 0.05
        ces_score = 0.15
        
    turns = [
        {'speaker': 0, 'text': text_t1},   # Customer Turn 1
        {'speaker': 1, 'text': agent_reply}, # Agent Turn 2
        {'speaker': 0, 'text': text_t3}    # Customer Turn 3
    ]
    
    return turns, spans_t1, traj_label, esc_score, ces_score

class HierarchicalDialogueDataset(Dataset):
    def __init__(self, df, tokenizer, max_turns=3, max_turn_len=48):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_turns = max_turns
        self.max_turn_len = max_turn_len
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        turns, spans_t1, traj_label, esc_score, ces_score = build_multi_turn_dialogue(row)
        
        turn_input_ids = []
        turn_attention_masks = []
        speaker_ids = []
        
        for t in turns[:self.max_turns]:
            enc = self.tokenizer(
                t['text'],
                padding='max_length',
                truncation=True,
                max_length=self.max_turn_len,
                return_tensors='pt'
            )
            turn_input_ids.append(enc['input_ids'].squeeze(0))
            turn_attention_masks.append(enc['attention_mask'].squeeze(0))
            speaker_ids.append(t['speaker'])
            
        flags_vec = torch.tensor([row[f'flag_{f}'] for f in FLAGS], dtype=torch.float)
        
        return {
            'turn_input_ids': torch.stack(turn_input_ids),         # [Num_Turns, Max_Len]
            'turn_attention_mask': torch.stack(turn_attention_masks), # [Num_Turns, Max_Len]
            'speaker_ids': torch.tensor(speaker_ids, dtype=torch.long), # [Num_Turns]
            'category_label': torch.tensor(cat2id[row['category']], dtype=torch.long),
            'intent_label': torch.tensor(intent2id[row['intent']], dtype=torch.long),
            'flags_label': flags_vec,
            'trajectory_label': torch.tensor(traj_label, dtype=torch.long),
            'escalation_label': torch.tensor(esc_score, dtype=torch.float),
            'effort_label': torch.tensor(ces_score, dtype=torch.float)
        }

## 3. Hierarchical Turn-Transformer Architecture (HAT + Telemetry Heads)

In [ ]:
class HierarchicalCustomerSupportTransformer(nn.Module):
    def __init__(
        self,
        model_name=MODEL_NAME,
        num_categories=len(CATEGORIES),
        num_intents=len(INTENTS),
        num_flags=len(FLAGS),
        num_trajectories=len(TRAJECTORIES),
        pos_weights=FLAG_POS_WEIGHTS,
        turn_layers=2,
        dropout_rate=0.2
    ):
        super().__init__()
        
        # =========================================================================
        # LEVEL 1: Sentence-Level Utterance Encoder (DistilBERT)
        # =========================================================================
        self.sentence_encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.sentence_encoder.config.hidden_size # 768
        self.dropout = nn.Dropout(dropout_rate)
        
        # =========================================================================
        # LEVEL 2: Dialogue-Level Turn-Transformer (HAT with Cross-Turn Attention)
        # =========================================================================
        turn_encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=8,
            dim_feedforward=hidden_size * 2,
            dropout=dropout_rate,
            activation='gelu',
            batch_first=True
        )
        self.turn_transformer = nn.TransformerEncoder(turn_encoder_layer, num_layers=turn_layers)
        self.speaker_embedding = nn.Embedding(num_embeddings=2, embedding_dim=hidden_size) # 0=Cust, 1=Agent
        
        # =========================================================================
        # LEVEL 3: PERCEPTION & TELEMETRY HEADS
        # =========================================================================
        # Dialogue Context-Aware Category & Intent (Powered by Turn-Transformer d_t)
        self.category_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size // 2, num_categories)
        )
        self.intent_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size // 2, num_intents)
        )
        
        # Instantaneous Utterance Emotion Flags (Powered by u_latest)
        self.flags_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size // 2, num_flags)
        )
        
        # Multi-Turn Trajectory Heads (Powered by Turn-Transformer d_t)
        self.trajectory_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size // 2, num_trajectories)
        )
        self.escalation_head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.GELU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        self.effort_head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.GELU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        # Loss functions
        self.ce_loss = nn.CrossEntropyLoss()
        self.flags_bce_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weights.to(DEVICE))
        self.smooth_l1 = nn.SmoothL1Loss()

    def forward(
        self,
        turn_input_ids,         # [Batch, Num_Turns, Max_Len]
        turn_attention_mask,    # [Batch, Num_Turns, Max_Len]
        speaker_ids,            # [Batch, Num_Turns]
        category_label=None,
        intent_label=None,
        flags_label=None,
        trajectory_label=None,
        escalation_label=None,
        effort_label=None
    ):
        batch_size, num_turns, max_len = turn_input_ids.shape
        
        # Collapse batch and turns to pass through DistilBERT in parallel
        flat_input_ids = turn_input_ids.view(batch_size * num_turns, max_len)
        flat_attention_mask = turn_attention_mask.view(batch_size * num_turns, max_len)
        
        encoder_out = self.sentence_encoder(input_ids=flat_input_ids, attention_mask=flat_attention_mask)
        flat_cls = encoder_out.last_hidden_state[:, 0, :] # [Batch * Num_Turns, 768]
        
        # Reshape to sequence of turns [Batch, Num_Turns, 768]
        turn_vectors = flat_cls.view(batch_size, num_turns, -1)
        turns_with_speakers = turn_vectors + self.speaker_embedding(speaker_ids)
        
        # Level 2: Dialogue-Level Cross-Turn Attention
        dialogue_context = self.turn_transformer(turns_with_speakers) # [Batch, Num_Turns, 768]
        d_t = self.dropout(dialogue_context[:, -1, :])               # Contextualized conversation state
        u_latest = self.dropout(turn_vectors[:, -1, :])              # Latest utterance state
        
        # Category & Intent are Context-Aware (Powered by d_t)
        cat_logits = self.category_head(d_t)
        intent_logits = self.intent_head(d_t)
        
        # Emotion Flags are instantaneous on current utterance (Powered by u_latest)
        flags_logits = self.flags_head(u_latest)
        
        # Trajectory, Escalation, Effort from dialogue context (Powered by d_t)
        traj_logits = self.trajectory_head(d_t)
        escalation_pred = self.escalation_head(d_t).squeeze(-1)
        effort_pred = self.effort_head(d_t).squeeze(-1)
        
        total_loss = None
        losses = {}
        
        if category_label is not None:
            l_cat = self.ce_loss(cat_logits, category_label)
            l_intent = self.ce_loss(intent_logits, intent_label)
            l_flags = self.flags_bce_loss(flags_logits, flags_label)
            l_traj = self.ce_loss(traj_logits, trajectory_label)
            l_esc = self.smooth_l1(escalation_pred, escalation_label)
            l_eff = self.smooth_l1(effort_pred, effort_label)
            
            total_loss = (
                1.0 * l_intent +
                0.5 * l_cat +
                1.0 * l_flags +
                1.5 * l_traj +
                1.2 * l_esc +
                1.0 * l_eff
            )
            
            losses = {
                'total': total_loss.item(),
                'intent': l_intent.item(),
                'trajectory': l_traj.item(),
                'escalation': l_esc.item(),
                'effort': l_eff.item()
            }
            
        return {
            'loss': total_loss,
            'losses': losses,
            'cat_logits': cat_logits,
            'intent_logits': intent_logits,
            'flags_logits': flags_logits,
            'traj_logits': traj_logits,
            'escalation_pred': escalation_pred,
            'effort_pred': effort_pred
        }

## 4. Train / Validation Split & DataLoaders

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['category'])
print(f'Train dialogues: {len(train_df):,} | Val dialogues: {len(val_df):,}')

train_dataset = HierarchicalDialogueDataset(train_df, tokenizer, max_turns=3, max_turn_len=48)
val_dataset = HierarchicalDialogueDataset(val_df, tokenizer, max_turns=3, max_turn_len=48)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

model = HierarchicalCustomerSupportTransformer().to(DEVICE)
print(f'Hierarchical NLU Model Initialized with {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters.')

## 5. GPU Training Loop (5 Epochs + Early Stopping & Best Checkpoint Saver)

In [ ]:
EPOCHS = 5
PATIENCE = 2
BEST_MODEL_PATH = 'best_hierarchical_nlu.pt'

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

best_val_loss = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')
    
    for batch in pbar:
        optimizer.zero_grad()
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        
        out = model(**batch)
        loss = out['loss']
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        total_train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation
    model.eval()
    total_val_loss = 0.0
    correct_intent = 0
    correct_traj = 0
    total_samples = 0
    esc_errors = []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]'):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            
            total_val_loss += out['loss'].item()
            pred_intent = out['intent_logits'].argmax(dim=-1)
            pred_traj = out['traj_logits'].argmax(dim=-1)
            
            correct_intent += (pred_intent == batch['intent_label']).sum().item()
            correct_traj += (pred_traj == batch['trajectory_label']).sum().item()
            total_samples += batch['intent_label'].size(0)
            esc_errors.extend(torch.abs(out['escalation_pred'] - batch['escalation_label']).cpu().numpy())
            
    avg_val_loss = total_val_loss / len(val_loader)
    val_intent_acc = correct_intent / total_samples
    val_traj_acc = correct_traj / total_samples
    val_esc_mae = np.mean(esc_errors)
    
    print(f'\n--- Epoch {epoch+1} Results ---')
    print(f'Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')
    print(f'Val Intent Accuracy:     {val_intent_acc * 100:.2f}%')
    print(f'Val Trajectory Accuracy: {val_traj_acc * 100:.2f}%')
    print(f'Val Escalation MAE:      {val_esc_mae:.4f}')
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f'🌟 [CHECKPOINT] Saved best hierarchical model to {BEST_MODEL_PATH}')
    else:
        patience_counter += 1
        print(f'⚠️ [PATIENCE] No improvement for {patience_counter}/{PATIENCE} epoch(s).')
        if patience_counter >= PATIENCE:
            print(f'🛑 [EARLY STOPPING] Triggered at Epoch {epoch+1}.')
            break
    print('-' * 45)

print(f'\nRestoring best model checkpoint from {BEST_MODEL_PATH}...')
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
model.eval()
print('✅ Hierarchical Model Ready for Multi-Turn Inference!')

## 6. Real-Time Multi-Turn Telemetry Engine (Emits JSON for LLM Agents)

In [ ]:
def extract_slots_from_text(text):
    slots = {}
    order_match = re.search(r'\b(ORD-\d+|#\d{5,8}|PO-\d+)\b', text, re.IGNORECASE)
    if order_match: slots['Order_Number'] = order_match.group(1)
    amount_match = re.search(r'(\$|€|£)\s*\d+(\.\d{2})?', text)
    if amount_match: slots['Refund_Amount'] = amount_match.group(0)
    inv_match = re.search(r'\b(INV-\d+)\b', text, re.IGNORECASE)
    if inv_match: slots['Invoice_Number'] = inv_match.group(1)
    return slots

def predict_dialogue_trajectory(conversation_turns):
    """
    Returns a clean, structured telemetry JSON payload for downstream LLM Agents.
    """
    model.eval()
    
    turn_input_ids = []
    turn_attention_masks = []
    speaker_ids = []
    
    for t in conversation_turns[-5:]: # Lookback window of up to 5 turns
        spk = 0 if 'customer' in t['speaker'].lower() else 1
        enc = tokenizer(
            t['text'],
            padding='max_length',
            truncation=True,
            max_length=48,
            return_tensors='pt'
        )
        turn_input_ids.append(enc['input_ids'].squeeze(0))
        turn_attention_masks.append(enc['attention_mask'].squeeze(0))
        speaker_ids.append(spk)
        
    input_ids_tensor = torch.stack(turn_input_ids).unsqueeze(0).to(DEVICE)
    attention_mask_tensor = torch.stack(turn_attention_masks).unsqueeze(0).to(DEVICE)
    speaker_ids_tensor = torch.tensor(speaker_ids, dtype=torch.long).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        out = model(
            turn_input_ids=input_ids_tensor,
            turn_attention_mask=attention_mask_tensor,
            speaker_ids=speaker_ids_tensor
        )
        
    cat_idx = out['cat_logits'].argmax(dim=-1).item()
    intent_idx = out['intent_logits'].argmax(dim=-1).item()
    traj_idx = out['traj_logits'].argmax(dim=-1).item()
    
    esc_score = out['escalation_pred'].item()
    effort_score = out['effort_pred'].item()
    
    flag_probs = torch.sigmoid(out['flags_logits']).squeeze(0).cpu().numpy()
    traj_probs = torch.softmax(out['traj_logits'], dim=-1).squeeze(0).cpu().numpy()
    traj_distribution = {id2traj[i]: f"{round(float(p) * 100, 1)}%" for i, p in enumerate(traj_probs)}
    
    all_text = ' '.join([t['text'] for t in conversation_turns])
    extracted_entities = extract_slots_from_text(all_text)
    
    return {
        'intent': id2intent[intent_idx],
        'category': id2cat[cat_idx],
        'entities': extracted_entities,
        'trajectory_state': id2traj[traj_idx],
        'dynamic_escalation': round(esc_score, 3),
        'customer_effort_score': round(effort_score, 3),
        'current_emotion_profile': {
            'Anger': f"{round(float(flag_probs[flag2id['W']]) * 100, 1)}%",
            'Frustration': f"{round(float(flag_probs[flag2id['M']]) * 100, 1)}%",
            'Distress': f"{round(float(flag_probs[flag2id['E']]) * 100, 1)}%",
            'Politeness': f"{round(float(flag_probs[flag2id['I']]) * 100, 1)}%"
        },
        'trajectory_distribution': traj_distribution
    }

## 7. Extended Multi-Turn Test Suite (Simulated Trajectories)

In [ ]:
test_dialogues = [
    {
        'title': 'Case 1: Severe Escalation (Swearing & Order Tracking Friction)',
        'turns': [
            {'speaker': 'Customer', 'text': 'I ordered 2 days ago, order ORD-88192, where is it?'},
            {'speaker': 'Agent',    'text': 'Please confirm your account email address so I can check.'},
            {'speaker': 'Customer', 'text': 'I already gave it twice! Stop asking stupid questions, cancel it damn it!'}
        ]
    },
    {
        'title': 'Case 2: Routine Order Tracking (Stable Continuation)',
        'turns': [
            {'speaker': 'Customer', 'text': 'Hello, could you help me check the tracking status of order ORD-55421?'},
            {'speaker': 'Agent',    'text': 'Sure! Could you verify your delivery city?'},
            {'speaker': 'Customer', 'text': 'Yes, the delivery city is Chicago. Thanks for checking!'}
        ]
    },
    {
        'title': 'Case 3: Financial Refund Request (Double Charge)',
        'turns': [
            {'speaker': 'Customer', 'text': 'I was charged twice $89.99 for order ORD-11234.'},
            {'speaker': 'Agent',    'text': 'We are looking into your billing records. Please hold on.'},
            {'speaker': 'Customer', 'text': 'I need this refund processed immediately please, I cannot afford this.'}
        ]
    }
]

print('=' * 85)
print('MULTI-TURN HIERARCHICAL TELEMETRY EVALUATION')
print('=' * 85)

for case in test_dialogues:
    print(f"\n📁 {case['title']}")
    for t in case['turns']:
        print(f"   [{t['speaker']}]: \"{t['text']}\"")
        
    telemetry = predict_dialogue_trajectory(case['turns'])
    badge = '🚨 [CRITICAL ALERT]' if telemetry['dynamic_escalation'] > 0.40 else '✅ [NORMAL]'
    
    print(f"\n   📦 Structured Telemetry Output (Sent to LLM Agent):")
    print(f"      ├─ Intent          : {telemetry['intent']}")
    print(f"      ├─ Category        : {telemetry['category']}")
    print(f"      ├─ Extracted Slots : {telemetry['entities']}")
    print(f"      ├─ Trajectory State: 🎯 {telemetry['trajectory_state']}")
    print(f"      ├─ Escalation Index: {telemetry['dynamic_escalation']} {badge}")
    print(f"      ├─ Customer Effort : {telemetry['customer_effort_score']}")
    print(f"      └─ Emotions        : {telemetry['current_emotion_profile']}")
    print('-' * 85)

## 8. Real Twitter Customer Support Threads Test (`twcs.csv` Zero-Shot Evaluation)

In [ ]:
TWITTER_THREADS_JSON = r'twitter_sample_threads.json'
if os.path.exists(TWITTER_THREADS_JSON):
    with open(TWITTER_THREADS_JSON, 'r', encoding='utf-8') as f:
        twitter_threads = json.load(f)
        
    print('=' * 85)
    print('REAL TWITTER THREADS ZERO-SHOT EVALUATION')
    print('=' * 85)
    
    for i, thread in enumerate(twitter_threads[:3]):
        print(f"\n🐦 TWITTER THREAD {i+1} ({len(thread)} Turns):")
        formatted_history = []
        for turn in thread:
            spk = 'Customer' if turn['is_customer'] else 'Agent'
            formatted_history.append({'speaker': spk, 'text': turn['text']})
            print(f"   [{spk}]: \"{turn['text']}\"")
            
        res = predict_dialogue_trajectory(formatted_history)
        print(f"\n   🎯 Trajectory State : {res['trajectory_state']}")
        print(f"   🚨 Escalation Index : {res['dynamic_escalation']}")
        print(f"   📊 Extracted Slots  : {res['entities']}")
        print('-' * 85)
else:
    print('Run build_twitter_threads.py first to generate twitter_sample_threads.json')